# Model inference

In [1]:
%%capture
from pathlib import Path

if Path.cwd().name == "notebooks_altair":
    %cd ..

%load_ext autoreload
%autoreload 2

In [2]:
import tomllib
from pathlib import Path

import altair as alt
import joblib

from src.data.database_manager import DatabaseManager
from src.plots_altair import plot_model_inference, style_figure

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
configuration_path = Path("src/experiments/measurement/measurement_config.toml")
with configuration_path.open("rb") as file:
    stimulus_seeds = tomllib.load(file)["stimulus"]["seeds"]

feature_set = "eda_raw_heart_rate_pupil"
inference_step_size_ms = 250
seed_key = "_".join(map(str, stimulus_seeds))
cache_path = Path(".cache/model_inference") / (
    f"{feature_set}_inference_probabilities_"
    f"seeds_{seed_key}_step_size_{inference_step_size_ms}.joblib"
)
model_inference_results = joblib.load(cache_path)

database = DatabaseManager()
with database:
    ratings = database.get_table("Feature_Data")

In [6]:
model_inference_chart = plot_model_inference(
    model_inference_results["probabilities"],
    sample_duration_ms=model_inference_results["sample_duration_ms"],
    classification_threshold=0.9,
    step_size_ms=inference_step_size_ms,
    display_step_size_ms=250,
    seeds_to_plot=stimulus_seeds,
    only_decreases=True,
    only_non_decreases=False,
    ncols=2,
    width=500,
    height=150,
    stimulus_scale=0.5,
    stimulus_linewidth=1.5,
    ratings_df=ratings,
    column_spacing=20,
    row_spacing=10,
    panel_border_color="#606060",
    panel_border_width=0.8,
    title=None,
)
style_figure(model_inference_chart)

alt.HConcatChart(...)

In [7]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
figure_dir = Path(os.environ["FIGURE_DIR"])
figure_dir.mkdir(parents=True, exist_ok=True)
style_figure(model_inference_chart).save(
    figure_dir / "model_inference_eda_raw_heart_rate_pupil.svg"
)

For the paper, we need…

Main section
- EDA, HR, and pupil, threshold 0.9

Supplementary section
- EDA, threshold 0.85
- HR, threshold 0.60
- Pupil, threshold 0.5
- EDA + HR, threshold 0.85
- EDA + Pupil, threshold 0.9
- Facial Expressions, threshold 0.7
- Combined w/o EEG, threshold 0.9
- EEG, threshold 0.65
- EEG + EDA, threshold 0.9
- Combined, threshold 0.9
- EDA + HR + Pupil for Non-decreases, threshold 0.9
- EDA + HR + Pupil for Decreases and non-decreases, threshold 0.5


In [15]:
face_feature_set = "brow_furrow_cheek_raise_mouth_open_nose_wrinkle_upper_lip_raise"
eeg_feature_set = "f3_f4_c3_cz_c4_p3_p4_oz"

paper_inference_plots = [
    {
        "feature_set": "eda_raw_heart_rate_pupil",
        "classification_threshold": 0.9,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": "model_inference_eda_raw_heart_rate_pupil.svg",
    },
    {
        "feature_set": "eda_raw",
        "classification_threshold": 0.85,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": "model_inference_eda_raw.svg",
    },
    {
        "feature_set": "heart_rate",
        "classification_threshold": 0.6,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": "model_inference_heart_rate.svg",
    },
    {
        "feature_set": "pupil",
        "classification_threshold": 0.5,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": "model_inference_pupil.svg",
    },
    {
        "feature_set": "eda_raw_heart_rate",
        "classification_threshold": 0.85,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": "model_inference_eda_raw_heart_rate.svg",
    },
    {
        "feature_set": "eda_raw_pupil",
        "classification_threshold": 0.9,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": "model_inference_eda_raw_pupil.svg",
    },
    {
        "feature_set": face_feature_set,
        "classification_threshold": 0.7,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": f"model_inference_{face_feature_set}.svg",
    },
    {
        "feature_set": f"{face_feature_set}_eda_raw_heart_rate_pupil",
        "classification_threshold": 0.9,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": (
            f"model_inference_{face_feature_set}_eda_raw_heart_rate_pupil.svg"
        ),
    },
    {
        "feature_set": eeg_feature_set,
        "classification_threshold": 0.65,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": f"model_inference_{eeg_feature_set}.svg",
    },
    {
        "feature_set": f"{eeg_feature_set}_eda_raw",
        "classification_threshold": 0.9,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": f"model_inference_{eeg_feature_set}_eda_raw.svg",
    },
    {
        "feature_set": (
            f"{eeg_feature_set}_{face_feature_set}_eda_raw_heart_rate_pupil"
        ),
        "classification_threshold": 0.9,
        "only_decreases": True,
        "only_non_decreases": False,
        "output_filename": (
            f"model_inference_{eeg_feature_set}_{face_feature_set}"
            "_eda_raw_heart_rate_pupil.svg"
        ),
    },
    {
        "feature_set": "eda_raw_heart_rate_pupil",
        "classification_threshold": 0.1,
        "only_decreases": False,
        "only_non_decreases": True,
        "output_filename": "model_inference_non_decreases.svg",
    },
    {
        "feature_set": "eda_raw_heart_rate_pupil",
        "classification_threshold": 0.5,
        "only_decreases": False,
        "only_non_decreases": False,
        "output_filename": "model_inference_50.svg",
    },
]

for inference_plot in paper_inference_plots:
    cache_path = Path(".cache/model_inference") / (
        f"{inference_plot['feature_set']}_inference_probabilities_"
        f"seeds_{seed_key}_step_size_{inference_step_size_ms}.joblib"
    )
    inference_results = joblib.load(cache_path)
    chart = plot_model_inference(
        inference_results["probabilities"],
        sample_duration_ms=inference_results["sample_duration_ms"],
        classification_threshold=inference_plot["classification_threshold"],
        step_size_ms=inference_step_size_ms,
        display_step_size_ms=250,
        seeds_to_plot=stimulus_seeds,
        only_decreases=inference_plot["only_decreases"],
        only_non_decreases=inference_plot["only_non_decreases"],
        ncols=2,
        width=500,
        height=150,
        stimulus_scale=0.5,
        stimulus_linewidth=1.5,
        ratings_df=ratings,
        column_spacing=20,
        row_spacing=10,
        panel_border_color="#606060",
        panel_border_width=0.8,
        title=None,
    )
    style_figure(chart).save(figure_dir / inference_plot["output_filename"])
